# IRWA 2026 - Part 2: Indexing and Evaluation

This notebook implements an inverted index and a search engine using
the processed corpus from Part 1.

The search engine uses conjunctive queries (AND) and ranks the matching
documents using TF-IDF scores.

We also implement several evaluation metrics to measure the performance
of the search engine using the provided validation labels and five
additional test queries.

See `README.md` for run instructions.

## 1. Load the processed corpus

Loads the documents pre-processed in Part 1 from
`data/processed_corpus.jsonl`.

The same text preprocessing function is used for both
documents and search queries.

In [95]:
from pathlib import Path
import sys
from IPython.display import display

# Find the repository root
ROOT = Path.cwd().resolve()

while not (ROOT / "web_app.py").exists():
    if ROOT == ROOT.parent:
        raise RuntimeError("Repository root not found")
    ROOT = ROOT.parent

sys.path.insert(0, str(ROOT))

from myapp.search.preprocessing import (
    load_processed_corpus,
    build_terms
)

preprocess_query = build_terms

#Load the processed documents from Part 1
DATA_PATH = ROOT / "data" / "processed_corpus.jsonl"
docs = load_processed_corpus(DATA_PATH)
print("Number of documents:", len(docs))
print("Example query:", preprocess_query("men slim jeans blue"))

Number of documents: 28080
Example query: ['men', 'slim', 'jean', 'blue']


## 2. Inverted index

Builds an inverted index using the terms stored in `all_terms`.

For each term, the index stores the document IDs where it appears
and its frequency in each document.

In [96]:
from collections import defaultdict, Counter

def build_inverted_index(docs):
    inverted_index = defaultdict(dict)
    for pid, row in docs.iterrows():
        terms = row["all_terms"]
        term_frequencies = Counter(terms) #Stores how much each word appears
        for term, frequency in term_frequencies.items():
            inverted_index[term][pid] = frequency #Creates the inverted index with the data obtained
    return dict(inverted_index)

In [97]:
# Here we will build our inverted index
inverted_index = build_inverted_index(docs)
print("Number of documents:", len(docs))
print("Number of unique terms:", len(inverted_index))
# term = "jean"
# print(f"\nDocuments containing '{term}':",len(inverted_index[term]))

Number of documents: 28080
Number of unique terms: 6822


## 3. Conjunctive search and TF-IDF ranking

### 3.1 Conjunctive search (AND)

The search engine retrieves only documents containing all the
processed query terms.

The matching documents are obtained by intersecting the posting
lists of the query terms.

In [98]:
def and_search(query, inverted_index):
    query_terms = preprocess_query(query)
    query_terms = list(dict.fromkeys(query_terms))
    if not query_terms:
        return []
    for term in query_terms:
        if term not in inverted_index:
            return []
    matching_docs = set(inverted_index[query_terms[0]])
    for term in query_terms[1:]:
        matching_docs.intersection_update(
            inverted_index[term]
        )
    return sorted(matching_docs)

### 3.2 TF-IDF ranking

The matching documents are ranked using TF-IDF scores.

Term frequency measures how often a query term appears in a document,
while inverse document frequency gives more importance to terms
that are less common in the corpus.

Documents with higher scores appear first in the ranking.

In [99]:
import math

def tfidf_ranking(query, docs, inverted_index):
    query_terms = list(dict.fromkeys(preprocess_query(query)))#preprocess the query
    matching_docs = and_search(query, inverted_index)
    if not matching_docs:
        return []
    N = len(docs)# total number of documents
    idf = {}
    for term in query_terms:
        df = len(inverted_index[term])
        idf[term] = math.log((N + 1) / (df + 1)) + 1

    ranked_results = []
    for pid in matching_docs:
        score = 0
        for term in query_terms:
            tf = inverted_index[term][pid]
            tf_weight = 1 + math.log(tf)# Logarithmic TF
            score += tf_weight * idf[term]

        ranked_results.append((pid, score))
    ranked_results.sort(key=lambda x: (-x[1], x[0]))#Sort by score, highest first

    return ranked_results

In [100]:
test_queries = {
    "Q1": "men cotton casual shirt",
    "Q2": "women black round neck tshirt",
    "Q3": "men blue track pants",
    "Q4": "women printed kurta",
    "Q5": "women winter jacket"
}

for query_id, query_text in test_queries.items():

    ranked = tfidf_ranking(
        query_text,
        docs,
        inverted_index
    )
    print(f"{query_id}: {query_text}")
    print(f"Matching documents: {len(ranked)}")
    print("---------------------------------------")

Q1: men cotton casual shirt
Matching documents: 1442
---------------------------------------
Q2: women black round neck tshirt
Matching documents: 1189
---------------------------------------
Q3: men blue track pants
Matching documents: 242
---------------------------------------
Q4: women printed kurta
Matching documents: 42
---------------------------------------
Q5: women winter jacket
Matching documents: 474
---------------------------------------


In [101]:
import pandas as pd
def show_search_results(query, k=10):
    ranked = tfidf_ranking(
        query,
        docs,
        inverted_index
    )
    top_results = ranked[:k]
    result_table = pd.DataFrame(
        top_results,
        columns=["pid", "tfidf_score"]
    )
    result_table["title"] = result_table["pid"].map(
        docs["title"]
    )
    result_table["sub_category"] = result_table["pid"].map(
        docs["sub_category"]
    )
    result_table["tfidf_score"] = (
        result_table["tfidf_score"].round(3)
    )
    return result_table

In [102]:
show_search_results(test_queries["Q1"])

,pid,tfidf_score,title,sub_category
0,SHTFMHXV84URWXKH,21.640,Men Regular Fit Solid Button Down Collar Casua...,Topwear
1,SHTFACSABCZFFTVE,21.014,Men Slim Fit Solid Mandarin Collar Casual Shirt,Topwear
2,SHTFACSAS6WTFZ5G,21.014,Men Slim Fit Solid Mandarin Collar Casual Shirt,Topwear
3,SHTFACSAUKWYMJYP,21.014,Men Slim Fit Solid Mandarin Collar Casual Shirt,Topwear
4,SHTFACSAYCSXZKDA,21.014,Men Slim Fit Solid Mandarin Collar Casual Shirt,Topwear
5,SHTFGXTMRYDSGPZE,20.934,Men Slim Fit Solid Mandarin Collar Casual Shirt,Topwear
6,SHTFGHEUUQKQZYQG,20.331,Men Slim Fit Solid Mandarin Collar Casual Shirt,Topwear
7,SHTFGXTMKYVVYCGP,20.331,Men Slim Fit Solid Mandarin Collar Casual Shirt,Topwear
8,SHTFGXTMQ2ZD8RZM,20.331,Men Slim Fit Solid Mandarin Collar Casual Shirt,Topwear
9,SHTFHFHHCVYUJMYY,20.331,Men Slim Fit Solid Button Down Collar Casual S...,Topwear


## 4. Evaluation metrics

In [103]:
def precision_at_k(ranked_pids, relevant_pids, k):
    top_k = ranked_pids[:k]

    relevant_found = sum(
        pid in relevant_pids for pid in top_k
    )
    return relevant_found / k

In [104]:
def recall_at_k(ranked_pids, relevant_pids, k):
    if len(relevant_pids) == 0:
        return 0.0
    top_k = ranked_pids[:k]
    relevant_found = sum(
        pid in relevant_pids for pid in top_k
    )

    return relevant_found / len(relevant_pids)

In [105]:
def f1_at_k(ranked_pids, relevant_pids, k):
    precision = precision_at_k(
        ranked_pids, relevant_pids, k
    )
    recall = recall_at_k(
        ranked_pids, relevant_pids, k
    )
    if precision + recall == 0:
        return 0.0
    else:
        return 2 * (precision * recall) / (precision + recall)

In [106]:
def average_precision(ranked_pids, relevant_pids, k=None):
    if k is not None and k <= 0:
        raise ValueError("k must be positive")
    if len(relevant_pids) == 0:
        return 0.0
    limit = len(ranked_pids) if k is None else k
    relevant_found = 0
    precision_sum = 0.0
    for rank, pid in enumerate(ranked_pids[:limit], start=1):
        if pid in relevant_pids:
            relevant_found += 1
            precision_sum += relevant_found / rank
    if k is not None:
        denominator = min(len(relevant_pids), k)
    else:
        denominator = len(relevant_pids)

    return precision_sum / denominator

In [107]:
def mean_average_precision(query_rankings, query_relevance):
    if not query_rankings:
        return 0.0
    ap_scores = []
    for query_id, ranked_pids in query_rankings.items():
        relevant_pids = query_relevance[query_id]
        ap = average_precision(
            ranked_pids,
            relevant_pids
        )
        ap_scores.append(ap)
    return sum(ap_scores) / len(ap_scores)

### Mean Reciprocal Rank (MRR)

MRR evaluates how early the first relevant document appears in the ranking.
For each query, we calculate the reciprocal of the position of its first
relevant result. MRR is the average of these values across all queries.

In [108]:
def reciprocal_rank(ranked_pids, relevant_pids):
    for rank, pid in enumerate(ranked_pids, start=1):
        if pid in relevant_pids:
            return 1 / rank

    return 0.0
def mean_reciprocal_rank(query_rankings, query_relevance):
    if not query_rankings:
        return 0.0
    total = 0.0
    for query in query_rankings:
        total += reciprocal_rank(
            query_rankings[query],
            query_relevance[query]
        )

    return total / len(query_rankings)

### Normalized Discounted Cumulative Gain (NDCG@K)

NDCG measures ranking quality by giving higher importance to relevant
documents appearing at the top of the results.

We use binary relevance labels (1 for relevant and 0 for non-relevant)
and normalize the score using the ideal ranking.

In [109]:
def ndcg_at_k(ranked_pids, relevant_pids, k):
    dcg = 0.0

    for rank, pid in enumerate(ranked_pids[:k], start=1):
        relevance = 1 if pid in relevant_pids else 0
        dcg += relevance / math.log2(rank + 1)
    ideal_relevant = min(len(relevant_pids), k)
    idcg = sum(
        1 / math.log2(rank + 1)
        for rank in range(1, ideal_relevant + 1)
    )
    if idcg == 0:
        return 0.0

    return dcg / idcg

## 5. Evaluation using the validation dataset

We evaluate our search engine using the relevance judgments provided
in the validation_labels.csv file.

These labels allow us to compare the ranked search results against
the expected relevance of the products.

In [110]:
import pandas as pd

# Load validation labels
VALIDATION_PATH = ROOT / "data" / "validation_labels.csv"

validation_df = pd.read_csv(
    VALIDATION_PATH,
    dtype={"pid": str}
)

print("Number of validation records:", len(validation_df))
print("\nColumns:")
print(validation_df.columns.tolist())

print("\nFirst 10 records:")
display(validation_df.head(10))

Number of validation records: 40

Columns:
['title', 'pid', 'query_id', 'labels']

First 10 records:


,title,pid,query_id,labels
0,Full Sleeve Printed Women Sweatshirt,SWSFFVKBCQG5FHPF,1,1
1,Full Sleeve Striped Women Sweatshirt,SWSFJY5ZFHQ7HXKW,1,0
2,Full Sleeve Printed Women Sweatshirt,SWSFUY89NHMZHZPX,1,1
3,Full Sleeve Graphic Print Women Sweatshirt,SWSFXQ5YX6RZKHP4,1,1
4,Full Sleeve Solid Women Sweatshirt,JCKFTZBC3DMCVYXH,1,0
5,Full Sleeve Graphic Print Women Sweatshirt,SWSFWCTDHRABJFCR,1,1
6,Typography Women V Neck Multicolor T-Shirt (P...,TSHFUH9FABXHSDPJ,1,0
7,Full Sleeve Printed Women Sweatshirt,SWSFVGKENHBSVUWJ,1,1
8,Full Sleeve Solid Women Sweatshirt,JCKF7M8DNBB6WZ8Y,1,1
9,Full Sleeve Graphic Print Women Sweatshirt,SWSFWCSPEGMDH8FV,1,1


In [111]:
print("Number of documents per query:")
print(validation_df["query_id"].value_counts())

print("\nRelevance labels per query:")
print(
    validation_df.groupby("query_id")["labels"].value_counts()
)

Number of documents per query:
query_id
1    20
2    20
Name: count, dtype: int64

Relevance labels per query:
query_id  labels
1         1         13
          0          7
2         0         10
          1         10
Name: count, dtype: int64


### Evaluation queries and relevance judgments

We use the two predefined queries from the project instructions.

For each query, we compare the documents retrieved by our search engine
with the relevance judgments provided in the validation dataset.

In [112]:
validation_queries = {
    1: "women full sleeve sweatshirt cotton",
    2: "men slim jeans blue"
}

for query_id, query in validation_queries.items():

    query_labels = validation_df[
        validation_df["query_id"] == query_id
    ]

    judged_pids = set(query_labels["pid"])
    relevant_pids = set(
        query_labels[query_labels["labels"] == 1]["pid"]
    )
    ranked_results = tfidf_ranking(
        query,
        docs,
        inverted_index
    )

    ranked_pids = [pid for pid, score in ranked_results]
    retrieved_judged = judged_pids.intersection(ranked_pids)
    retrieved_relevant = relevant_pids.intersection(ranked_pids)

    print(f"Query {query_id}: {query}")
    print(f"Total retrieved documents: {len(ranked_pids)}")
    print(f"Judged documents: {len(judged_pids)}")
    print(f"Relevant documents: {len(relevant_pids)}")
    print(f"Retrieved judged documents: {len(retrieved_judged)}")
    print(f"Retrieved relevant documents: {len(retrieved_relevant)}")
    print("-----------------------------------" )

Query 1: women full sleeve sweatshirt cotton
Total retrieved documents: 500
Judged documents: 20
Relevant documents: 13
Retrieved judged documents: 20
Retrieved relevant documents: 13
-----------------------------------
Query 2: men slim jeans blue
Total retrieved documents: 224
Judged documents: 20
Relevant documents: 10
Retrieved judged documents: 8
Retrieved relevant documents: 6
-----------------------------------


### Preparing the evaluation rankings

Since the validation dataset contains relevance judgments for only
20 documents per query, we evaluate the ranking using the judged
documents.

We preserve the original TF-IDF ranking order and exclude unjudged
documents from the evaluation. Relevant documents that were not
retrieved are still considered when calculating recall.

This approach provides an evaluation on the judged subset, although
the results may not represent performance over the entire collection.

In [113]:
evaluation_rankings = {}
evaluation_relevance = {}

for query_id, query in validation_queries.items():

    query_labels = validation_df[
        validation_df["query_id"] == query_id
    ]
    judged_pids = set(query_labels["pid"])
    relevant_pids = set(
        query_labels[query_labels["labels"] == 1]["pid"]
    )

    ranked_results = tfidf_ranking(
        query,
        docs,
        inverted_index
    )

    ranked_pids = [pid for pid, score in ranked_results]
    evaluated_ranking = [
        pid for pid in ranked_pids
        if pid in judged_pids
    ]
    evaluation_rankings[query_id] = evaluated_ranking
    evaluation_relevance[query_id] = relevant_pids
    print(f"Query {query_id}: {query}")
    print(f"Documents in evaluation ranking: {len(evaluated_ranking)}")
    print(f"Relevant documents: {len(relevant_pids)}")
    print("---------------------------------")

Query 1: women full sleeve sweatshirt cotton
Documents in evaluation ranking: 20
Relevant documents: 13
---------------------------------
Query 2: men slim jeans blue
Documents in evaluation ranking: 8
Relevant documents: 10
---------------------------------


### 5.1 Evaluation results

We evaluate the two predefined queries using the relevance judgments
provided in the validation dataset.

The evaluation metrics are calculated at K=5 and K=10.
All results are rounded to three decimal places.

In [114]:
for k in [5, 10]:
    evaluation_results = []
    for query_id in validation_queries:

        ranked_pids = evaluation_rankings[query_id]
        relevant_pids = evaluation_relevance[query_id]

        results = {
            "Query": query_id,
            "Precision": precision_at_k(ranked_pids, relevant_pids, k),
            "Recall": recall_at_k(ranked_pids, relevant_pids, k),
            "F1": f1_at_k(ranked_pids, relevant_pids, k),
            "AP": average_precision(ranked_pids, relevant_pids, k),
            "NDCG": ndcg_at_k(ranked_pids, relevant_pids, k)
        }
        evaluation_results.append(results)
    results_df = pd.DataFrame(evaluation_results)
    results_df = results_df.set_index("Query")

    print(f"Evaluation at K={k}")
    display(results_df.style.format("{:.3f}"))

Evaluation at K=5


,Precision,Recall,F1,AP,NDCG
Query,,,,,
1,1.000,0.385,0.556,1.000,1.000
2,0.800,0.400,0.533,0.760,0.854


Evaluation at K=10


,Precision,Recall,F1,AP,NDCG
Query,,,,,
1,1.000,0.769,0.870,1.000,1.000
2,0.600,0.600,0.600,0.549,0.706


### 5.2 MAP and MRR

Mean Average Precision (MAP) is calculated as the average of
the Average Precision scores across both queries.

Mean Reciprocal Rank (MRR) measures the average reciprocal
position of the first relevant document.

In [115]:
map_score = mean_average_precision(
    evaluation_rankings,
    evaluation_relevance
)
mrr_score = mean_reciprocal_rank(
    evaluation_rankings,
    evaluation_relevance
)
overall_results = pd.DataFrame({
    "MAP": [map_score],
    "MRR": [mrr_score]
})
display(overall_results.style.format("{:.3f}"))

,MAP,MRR
0,0.766,1.000


## 6. Evaluation of our test queries

We evaluate the five test queries defined in Section 3.

For each query, we select the top 20 documents returned by the
search engine and manually assign binary relevance labels.

A document is labeled as 1 if it is relevant to the query,
and 0 otherwise.

In [116]:
EXPERT_DIR = ROOT / "project_progress" / "part_2" / "output"
EXPERT_DIR.mkdir(parents=True, exist_ok=True)
EXPERT_PATH = EXPERT_DIR / "expert_labels.csv"
if not EXPERT_PATH.exists():
    expert_rows = []
    for query_id, query in test_queries.items():
        ranked_results = tfidf_ranking(
            query,
            docs,
            inverted_index
        )
        for rank, (pid, score) in enumerate(
            ranked_results[:20], start=1
        ):
            document = docs.loc[pid]

            description = document["description"]
            if not isinstance(description, str):
                description = ""
            expert_rows.append({
                "query_id": query_id,
                "query": query,
                "rank": rank,
                "pid": pid,
                "title": document["title"],
                "sub_category": document["sub_category"],
                "description": description[:200],
                "relevance": ""
            })
    expert_df = pd.DataFrame(expert_rows)
    expert_df.to_csv(EXPERT_PATH, index=False)
    print("Expert labels file created.")

else:

    expert_df = pd.read_csv(EXPERT_PATH)

    print("Existing expert labels file found.")
    print("The file has not been overwritten.")

print("Number of documents:", len(expert_df))
print("File:", EXPERT_PATH)
display(expert_df.head(10))

Existing expert labels file found.
The file has not been overwritten.
Number of documents: 100
File: /Users/arnau/Documents/UNI/4t/IRWA/Final_project/riaw-search-engine/project_progress/part_2/output/expert_labels.csv


,query_id,query,rank,pid,title,sub_category,description,relevance
0,Q1,men cotton casual shirt,1,SHTFMHXV84URWXKH,Men Regular Fit Solid Button Down Collar Casua...,Topwear,"shirts,casual shirt ,half sleeve shirt,men par...",0
1,Q1,men cotton casual shirt,2,SHTFACSABCZFFTVE,Men Slim Fit Solid Mandarin Collar Casual Shirt,Topwear,Jai Textiles(U-TURN) Casual Shirt has a Mandar...,0
2,Q1,men cotton casual shirt,3,SHTFACSAS6WTFZ5G,Men Slim Fit Solid Mandarin Collar Casual Shirt,Topwear,Jai Textiles(U-TURN) Casual Shirt has a Mandar...,0
3,Q1,men cotton casual shirt,4,SHTFACSAUKWYMJYP,Men Slim Fit Solid Mandarin Collar Casual Shirt,Topwear,Jai Textiles(U-TURN) Casual Shirt has a Mandar...,0
4,Q1,men cotton casual shirt,5,SHTFACSAYCSXZKDA,Men Slim Fit Solid Mandarin Collar Casual Shirt,Topwear,Jai Textiles(U-TURN) Casual Shirt has a Mandar...,0
5,Q1,men cotton casual shirt,6,SHTFGXTMRYDSGPZE,Men Slim Fit Solid Mandarin Collar Casual Shirt,Topwear,A J Style Men's Cotton Casual Full Sleeves Sli...,1
6,Q1,men cotton casual shirt,7,SHTFGHEUUQKQZYQG,Men Slim Fit Solid Mandarin Collar Casual Shirt,Topwear,A J Style Men's Cotton Casual Full Sleeves Sli...,1
7,Q1,men cotton casual shirt,8,SHTFGXTMKYVVYCGP,Men Slim Fit Solid Mandarin Collar Casual Shirt,Topwear,A J Style Men's Cotton Casual Full Sleeves Sli...,1
8,Q1,men cotton casual shirt,9,SHTFGXTMQ2ZD8RZM,Men Slim Fit Solid Mandarin Collar Casual Shirt,Topwear,A J Style Men's Cotton Casual Full Sleeves Sli...,1
9,Q1,men cotton casual shirt,10,SHTFHFHHCVYUJMYY,Men Slim Fit Solid Button Down Collar Casual S...,Topwear,A J Style Men's Cotton Casual Full Sleeves Sli...,1


### 6.1 Manual relevance judgments

The relevance judgments were manually assigned by examining
the product information for each query.

We use binary relevance labels: 1 for relevant documents and
0 for non-relevant documents.

In [117]:
expert_df = pd.read_csv(
    EXPERT_PATH,
    dtype={"pid": str}
)

assert len(expert_df) == 100
assert expert_df["relevance"].notna().all()
assert expert_df["relevance"].isin([0, 1]).all()

print("Manual relevance labels verified!")

print("\nRelevance distribution:")
print(
    expert_df.groupby("query_id")["relevance"].value_counts()
)

Manual relevance labels verified!

Relevance distribution:
query_id  relevance
Q1        0            11
          1             9
Q2        1            18
          0             2
Q3        1            20
Q4        1            20
Q5        1            20
Name: count, dtype: int64


### 6.2 Preparing the evaluation rankings

For each test query, we retrieve the ranked documents using TF-IDF.

We keep the documents included in our manually labeled evaluation
set and use their relevance judgments to calculate the metrics.

In [118]:
custom_rankings = {}
custom_relevance = {}

for query_id, query in test_queries.items():
    query_labels = expert_df[
        expert_df["query_id"] == query_id
    ]
    judged_pids = set(query_labels["pid"])
    relevant_pids = set(
        query_labels[query_labels["relevance"] == 1]["pid"]
    )
    ranked_results = tfidf_ranking(
        query,
        docs,
        inverted_index
    )

    ranked_pids = [pid for pid, score in ranked_results]
    evaluated_ranking = [
        pid for pid in ranked_pids
        if pid in judged_pids
    ]
    custom_rankings[query_id] = evaluated_ranking
    custom_relevance[query_id] = relevant_pids

    print(f"{query_id}: {query}")
    print(f"Evaluated documents: {len(evaluated_ranking)}")
    print(f"Relevant documents: {len(relevant_pids)}")
    print("--------------------------")

Q1: men cotton casual shirt
Evaluated documents: 20
Relevant documents: 9
--------------------------
Q2: women black round neck tshirt
Evaluated documents: 20
Relevant documents: 18
--------------------------
Q3: men blue track pants
Evaluated documents: 20
Relevant documents: 20
--------------------------
Q4: women printed kurta
Evaluated documents: 20
Relevant documents: 20
--------------------------
Q5: women winter jacket
Evaluated documents: 20
Relevant documents: 20
--------------------------


### 6.3 Evaluation results for the test queries

We calculate Precision, Recall, F1-score, Average Precision and
NDCG at K=5 and K=10 for each of the five test queries.

The evaluation uses the manually assigned relevance labels
from the top 20 retrieved documents.

All results are rounded to three decimal places.

In [119]:
for k in [5, 10]:
    evaluation_results = []
    for query_id in test_queries:
        ranked_pids = custom_rankings[query_id]
        relevant_pids = custom_relevance[query_id]
        results = {
            "Query": query_id,
            "Precision": precision_at_k(ranked_pids, relevant_pids, k),
            "Recall": recall_at_k(ranked_pids, relevant_pids, k),
            "F1": f1_at_k(ranked_pids, relevant_pids, k),
            "AP": average_precision(ranked_pids, relevant_pids, k),
            "NDCG": ndcg_at_k(ranked_pids, relevant_pids, k)
        }
        evaluation_results.append(results)
    results_df = pd.DataFrame(evaluation_results)
    results_df = results_df.set_index("Query")
    print(f"Evaluation at K={k}")
    display(results_df.style.format("{:.3f}"))

Evaluation at K=5


,Precision,Recall,F1,AP,NDCG
Query,,,,,
Q1,0.000,0.000,0.000,0.000,0.000
Q2,0.600,0.167,0.261,0.287,0.447
Q3,1.000,0.250,0.400,1.000,1.000
Q4,1.000,0.250,0.400,1.000,1.000
Q5,1.000,0.250,0.400,1.000,1.000


Evaluation at K=10


,Precision,Recall,F1,AP,NDCG
Query,,,,,
Q1,0.500,0.556,0.526,0.197,0.375
Q2,0.800,0.444,0.571,0.514,0.641
Q3,1.000,0.500,0.667,1.000,1.000
Q4,1.000,0.500,0.667,1.000,1.000
Q5,1.000,0.500,0.667,1.000,1.000


### 6.4 MAP and MRR

MAP summarizes the average ranking quality across the five test
queries.

MRR measures how early the first relevant document appears
in the results of each query.

In [120]:
custom_map = mean_average_precision(
    custom_rankings,
    custom_relevance
)
custom_mrr = mean_reciprocal_rank(
    custom_rankings,
    custom_relevance
)
custom_overall_results = pd.DataFrame({
    "MAP": [custom_map],
    "MRR": [custom_mrr]
})
display(custom_overall_results.style.format("{:.3f}"))

,MAP,MRR
0,0.846,0.700


## 7. Discussion and limitations

### 7.1 Evaluation metrics

Each evaluation metric provides different information about the
performance of the search engine.

- **Precision@K:** Measures the proportion of relevant documents
  among the top K retrieved results.

- **Recall@K:** Measures the proportion of known relevant documents
  retrieved within the top K results.

- **F1-score@K:** Combines Precision and Recall into a single score
  using their harmonic mean.

- **Average Precision@K:** Evaluates the ranking by considering
  the precision at each position where a relevant document appears.

- **MAP:** Calculates the mean Average Precision across multiple
  queries, providing an overall measure of ranking quality.

- **MRR:** Measures how early the first relevant document appears
  in the ranking.

- **NDCG:** Evaluates ranking quality by giving more importance
  to relevant documents appearing in the first positions.

### 7.2 Results analysis

The evaluation results show differences in performance depending
on the query.

For Q1, Precision@5 is 0.000, meaning that none of the first
five evaluated products were considered relevant. However,
Precision@10 increases to 0.500, showing that relevant products
appear further down in the ranking.

For Q2, Precision@10 reaches 0.800, although the first two
evaluated documents are not relevant. This suggests that TF-IDF
retrieves relevant products but does not always place them
at the highest positions.

Q3, Q4 and Q5 achieve Precision@10 and NDCG@10 values of 1.000.
However, all 20 evaluated documents for these queries were
labeled as relevant, so these results should be interpreted
with caution.

The overall MAP is 0.846, while MRR is 0.700. These values
suggest that the ranking performs reasonably well within
the evaluated document sets.

One important limitation is that the evaluation only considers
the top 20 retrieved documents for each test query. Therefore,
the results do not necessarily represent the performance of
the search engine over the entire collection.

### 7.3 System limitations and possible improvements

Although the search engine retrieves relevant documents, there
are several limitations that could be improved.

**1. Strict AND queries**

Our search engine uses conjunctive queries, meaning that a document
must contain all query terms to be retrieved.

This can exclude relevant products that do not contain every
term in the query.

A possible improvement would be to support OR queries or more
flexible matching strategies.

**2. No field weighting**

The inverted index uses `all_terms`, combining terms from the
title, description, brand, category and other text fields.

Currently, all fields contribute equally to the TF-IDF score.
However, a term appearing in the title may be more important
than the same term appearing in the description or seller field.

A possible solution would be to implement field-weighted TF-IDF,
giving more importance to the title and product category.

**3. Limitations of TF-IDF ranking**

TF-IDF ranks documents based on term frequency and inverse
document frequency, but it does not understand the meaning
of the query.

It may also give higher scores to documents where query terms
appear repeatedly, even if they are not the most relevant.

Possible improvements include document-length normalization
or using a ranking algorithm such as BM25.

**4. Query formulation**

The search engine depends on the exact terms used in the query,
after preprocessing.

Synonyms and different ways of describing the same product
may lead to missing relevant results.

Query expansion or semantic search could help retrieve
documents with similar meanings.

**5. Indexing strategy**

The current inverted index stores term frequencies across
all text fields without distinguishing their positions.

A more advanced index could store separate postings for
different fields or term positions, allowing field-specific
and phrase-based searches.

**6. Evaluation limitations**

The validation dataset contains only 20 judged documents per
predefined query. For our five test queries, we evaluated only
the top 20 retrieved documents.

This limits the conclusions we can draw about the entire
collection of 28,080 products.

A possible improvement would be to evaluate more documents,
including results from different ranking positions, and use
additional human relevance judgments.

## 8. Conclusions

In this part of the project, we implemented an inverted index
using the processed corpus from Part 1.

We developed a search engine based on conjunctive queries and
TF-IDF ranking to retrieve and rank fashion products.

We also implemented several evaluation metrics, including
Precision, Recall, F1-score, Average Precision, MAP, MRR
and NDCG.

The search engine was evaluated using the two predefined
queries and five additional test queries.

The results show that TF-IDF can provide useful rankings,
although performance depends on the query and the relevance
of the retrieved documents.

The main limitations are related to strict AND matching,
the lack of field weighting and the limited number of
relevance judgments.

Future improvements could include more flexible queries,
field-weighted ranking and better evaluation datasets.